# APAN 5200 Machine Learning — Ensemble Models (Quiz 8)
## Answer Key
**Dataset:** Spotify Popular Songs  
**Outcome Variable:** `rating`

In [ ]:
# ============================================================
# VARIABLE DESCRIPTIONS
# ============================================================
# 0.  id                : Song ID
# 1.  performer         : Performer name
# 2.  song              : Song name
# 3.  genre             : Genre
# 4.  track_duration    : Duration in milliseconds
# 5.  track_explicit    : Is explicit
# 6.  danceability      : How suitable for dancing (0.0–1.0)
# 7.  energy            : Perceptual intensity/activity (0.0–1.0)
# 8.  key               : Estimated key of track (Pitch Class)
# 9.  loudness          : Overall loudness in dB
# 10. mode              : Major (1) or Minor (0)
# 11. speechiness       : Presence of spoken words (0.0–1.0)
# 12. acousticness      : Confidence of acoustic track (0.0–1.0)
# 13. instrumentalness  : Likelihood of no vocals (0.0–1.0)
# 14. liveness          : Presence of audience in recording (0.0–1.0)
# 15. valence           : Musical positiveness (0.0–1.0)
# 16. tempo             : Beats per minute
# 17. time_signature    : Time signature
# 18. rating            : TARGET — derived from Billboard Top 100 & Spotify popularity
# 19–28. genre_*        : Ten binary dummy variables derived from genre

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    VotingRegressor, BaggingRegressor,
    RandomForestRegressor, AdaBoostRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_squared_error

In [ ]:
# ============================================================
# Q1: Data Setup & Stratified Split
# 
# Read the data into an object called data.
# Drop: id, performer, song, genre, key.
# Separate outcome variable: y = rating, X = remaining features.
# Create binned outcome: y_binned = pd.qcut(data.rating, q=20)
# Stratified train/test split: 70% train, 30% test, random_state=1031
#
# QUESTION: What is the median value of liveness in the train sample?
# ANSWER  : 0.131
# ============================================================

data = pd.read_csv('spotify_data.csv')
data = data.drop(columns=['id', 'performer', 'song', 'genre', 'key'])

y = data['rating']
X = data.drop(columns=['rating'])

y_binned = pd.qcut(data.rating, q=20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1031, stratify=y_binned
)

print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])
print('Median liveness (train):', X_train['liveness'].median())

In [ ]:
# ============================================================
# Q2: Linear Regression — Genre Dummies Only
#
# Construct a linear regression model to predict rating using
# all ten genre_ dummy variables.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.8119
# ============================================================

genre_cols = [c for c in X.columns if c.startswith('genre_')]

lr = LinearRegression()
lr.fit(X_train[genre_cols], y_train)

rmse_lr = np.sqrt(mean_squared_error(y_train, lr.predict(X_train[genre_cols])))
print(f'Linear Regression RMSE (train): {rmse_lr:.4f}')

In [ ]:
# ============================================================
# Q3: Decision Tree — Genre Dummies Only
#
# Construct a tree model to predict rating using all ten genre_
# dummy variables. Use max_depth=4.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.6919
# ============================================================

tree = DecisionTreeRegressor(max_depth=4, random_state=1031)
tree.fit(X_train[genre_cols], y_train)

rmse_tree = np.sqrt(mean_squared_error(y_train, tree.predict(X_train[genre_cols])))
print(f'Tree RMSE (train): {rmse_tree:.4f}')

In [ ]:
# ============================================================
# Q4: Voting Model (LR + Tree)
#
# Combine the Linear Regression and Tree models above using
# a Voting model.
#
# QUESTION: What is the train sample RMSE?
# ANSWER  : 15.6708
# 
# KEY CONCEPT: VotingRegressor averages predictions from base
# learners — reduces variance without strong assumptions.
# The voting RMSE (15.67) falls BETWEEN the two base models,
# as expected.
# ============================================================

voting = VotingRegressor(estimators=[
    ('lr',   LinearRegression()),
    ('tree', DecisionTreeRegressor(max_depth=4, random_state=1031))
])
voting.fit(X_train[genre_cols], y_train)

rmse_voting = np.sqrt(mean_squared_error(y_train, voting.predict(X_train[genre_cols])))
print(f'Voting RMSE (train): {rmse_voting:.4f}')

In [ ]:
# ============================================================
# Q5: Bagging — Decision Tree Base Learner
#
# Construct a Bagging model with 100 bootstrapped samples.
# Fit regression tree models using ALL predictors.
# Set max_depth=10.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 12.8655
#
# KEY CONCEPT: Bagging with deeper trees uses ALL features at
# each split (unlike Random Forest which samples features).
# oob_score=True enables Out-of-Bag evaluation.
# ============================================================

bagging_tree = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=10),
    n_estimators=100,
    random_state=1031,
    oob_score=True
)
bagging_tree.fit(X_train, y_train)

rmse_bag_tree = np.sqrt(mean_squared_error(y_train, bagging_tree.predict(X_train)))
print(f'Bagging Tree RMSE (train): {rmse_bag_tree:.4f}')
print(f'Bagging Tree OOB R² score: {bagging_tree.oob_score_:.4f}')

In [ ]:
# ============================================================
# Q6: Bagging — Linear Regression Base Learner
#
# Construct a Bagging model with 100 bootstrapped samples,
# but fit a Linear Regression with ALL predictors.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.3495
#
# KEY INSIGHT: Bagging helps high-variance models (e.g., deep
# trees) more than low-variance models (e.g., linear regression).
# The improvement over plain LR is modest here.
# ============================================================

bagging_lr = BaggingRegressor(
    estimator=LinearRegression(),
    n_estimators=100,
    random_state=1031,
    oob_score=True
)
bagging_lr.fit(X_train, y_train)

rmse_bag_lr = np.sqrt(mean_squared_error(y_train, bagging_lr.predict(X_train)))
print(f'Bagging LR RMSE (train): {rmse_bag_lr:.4f}')
print(f'Bagging LR OOB R² score: {bagging_lr.oob_score_:.4f}')

In [ ]:
# ============================================================
# Q7: Random Forest
#
# Fit a Random Forest model with 100 bootstrapped samples.
# Use ALL predictors.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 5.6667
#
# KEY INSIGHT: RF introduces feature subsampling at each split
# (default: sqrt of features), which decorrelates trees and
# dramatically reduces variance vs. plain Bagging.
# ============================================================

rf = RandomForestRegressor(n_estimators=100, random_state=1031, oob_score=True)
rf.fit(X_train, y_train)

rmse_rf = np.sqrt(mean_squared_error(y_train, rf.predict(X_train)))
print(f'Random Forest RMSE (train): {rmse_rf:.4f}')
print(f'Random Forest OOB R² score: {rf.oob_score_:.4f}')

In [ ]:
# ============================================================
# Q8: Random Forest — Most Important Feature
#
# Based on the Random Forest model above, which feature is
# the most important?
#
# ANSWER: track_duration (importance ≈ 0.1489)
# ============================================================

feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns)\
             .sort_values(ascending=False)
print('Top 5 Feature Importances (Random Forest):')
print(feat_imp.head(5).to_string())

In [ ]:
# ============================================================
# Q9: Out-of-Bag (OOB) Score Comparison
#
# Which of the following models has the LOWEST OOB score?
#   (a) Bagging Tree  (b) Bagging LR  (c) Random Forest
#
# ANSWER: Bagging LR has the lowest OOB score (≈ 0.1433)
#
# NOTE: OOB score is the R² on the out-of-bag samples
# (samples not used in a given bootstrap iteration).
# A LOWER OOB score means WORSE generalization performance.
# ============================================================

print('OOB R² Scores:')
print(f'  Bagging Tree : {bagging_tree.oob_score_:.4f}')
print(f'  Bagging LR   : {bagging_lr.oob_score_:.4f}  ← LOWEST')
print(f'  Random Forest: {rf.oob_score_:.4f}')

In [ ]:
# ============================================================
# Q10: AdaBoost
#
# Construct a Boosting model using AdaBoost with a tree
# estimator. Use 100 estimators. Use ALL predictors.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 0.6612
#
# KEY INSIGHT: AdaBoost with DecisionTreeRegressor uses
# exponential reweighting of residuals. The train RMSE is
# very low (~0.66) — a sign of heavy overfitting to training
# data, especially with max_depth defaulting to 3.
# ============================================================

ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=100,
    random_state=1031
)
ada.fit(X_train, y_train)

rmse_ada = np.sqrt(mean_squared_error(y_train, ada.predict(X_train)))
print(f'AdaBoost RMSE (train): {rmse_ada:.4f}')

In [ ]:
# ============================================================
# Q11: Gradient Boosting
#
# Construct a Gradient Boosting model with 100 trees.
# Use ALL predictors.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 14.7569
#
# KEY CONCEPT: Gradient Boosting fits trees sequentially on
# the negative gradient (pseudo-residuals). Slower to overfit
# than AdaBoost with default settings.
# ============================================================

gb = GradientBoostingRegressor(n_estimators=100, random_state=1031)
gb.fit(X_train, y_train)

rmse_gb = np.sqrt(mean_squared_error(y_train, gb.predict(X_train)))
print(f'Gradient Boosting RMSE (train): {rmse_gb:.4f}')

In [ ]:
# ============================================================
# Q12: Stochastic Gradient Boosting
#
# Construct a Stochastic Gradient Boosting model with 100 trees.
# Set max_features=0.4 and subsample=0.6. Use ALL predictors.
#
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 14.7966
#
# KEY INSIGHT: Stochastic GB uses:
#   - subsample=0.6 → each tree trained on 60% of data (rows)
#   - max_features=0.4 → each split considers 40% of features
# This introduces randomness to reduce variance (like RF).
# ============================================================

sgb = GradientBoostingRegressor(
    n_estimators=100,
    max_features=0.4,
    subsample=0.6,
    random_state=1031
)
sgb.fit(X_train, y_train)

rmse_sgb = np.sqrt(mean_squared_error(y_train, sgb.predict(X_train)))
print(f'Stochastic GB RMSE (train): {rmse_sgb:.4f}')

In [ ]:
# ============================================================
# Q13: Stochastic Gradient Boosting — Most Important Feature
#
# ANSWER: track_duration (importance ≈ 0.2144)
# ============================================================

feat_imp_sgb = pd.Series(sgb.feature_importances_, index=X_train.columns)\
                 .sort_values(ascending=False)
print('Top 5 Feature Importances (Stochastic GB):')
print(feat_imp_sgb.head(5).to_string())

---
## 🧪 Bonus Practice Question 1
**Test Sample RMSE Comparison**

For each of the models trained above (Linear Regression with genre dummies, Bagging Tree, Random Forest, AdaBoost, Gradient Boosting, and Stochastic GB), compute the RMSE on the **test** sample. Which model achieves the lowest test RMSE?

In [ ]:
# ============================================================
# BONUS Q1: Test Sample RMSE for All Ensemble Models
#
# ANSWER: Random Forest has the lowest test RMSE.
# 
# KEY INSIGHT: Even though AdaBoost had the lowest TRAIN RMSE
# (~0.66), it overfits heavily. Random Forest generalises best.
# ============================================================

results = {
    'Linear Regression (genre)':   np.sqrt(mean_squared_error(y_test, lr.predict(X_test[genre_cols]))),
    'Bagging Tree':                 np.sqrt(mean_squared_error(y_test, bagging_tree.predict(X_test))),
    'Bagging LR':                   np.sqrt(mean_squared_error(y_test, bagging_lr.predict(X_test))),
    'Random Forest':                np.sqrt(mean_squared_error(y_test, rf.predict(X_test))),
    'AdaBoost':                     np.sqrt(mean_squared_error(y_test, ada.predict(X_test))),
    'Gradient Boosting':            np.sqrt(mean_squared_error(y_test, gb.predict(X_test))),
    'Stochastic GB':                np.sqrt(mean_squared_error(y_test, sgb.predict(X_test))),
}

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Test RMSE'])\
               .sort_values('Test RMSE')
print(results_df.to_string())
print(f"\nBest model: {results_df.idxmin()[0]} (RMSE = {results_df.min()[0]:.4f})")

---
## 🧪 Bonus Practice Question 2
**Random Forest Hyperparameter Tuning**

Train three Random Forest models with `n_estimators=100` but varying `max_features`: `'sqrt'` (default), `0.3`, and `0.7`. Compare their OOB R² scores. Which setting produces the highest OOB score?

In [ ]:
# ============================================================
# BONUS Q2: Random Forest — max_features Sensitivity
#
# KEY CONCEPT:
# - max_features controls feature subsampling per split.
# - Smaller values → more decorrelated trees → lower variance
#   but higher bias.
# - 'sqrt' is the standard default for classification;
#   for regression, 'sqrt' or 1/3 of features is common.
# ============================================================

oob_scores = {}
for mf in ['sqrt', 0.3, 0.7]:
    rf_mf = RandomForestRegressor(
        n_estimators=100,
        max_features=mf,
        random_state=1031,
        oob_score=True
    )
    rf_mf.fit(X_train, y_train)
    oob_scores[str(mf)] = rf_mf.oob_score_
    print(f'max_features={mf:>5} | OOB R² = {rf_mf.oob_score_:.4f}')

best_mf = max(oob_scores, key=oob_scores.get)
print(f'\nHighest OOB R² → max_features = {best_mf}')